# Notebook 05 — Re-train ConvNeXt-Base (recipe B) and keep its weights

**Why:** Notebook 02 saved weights only for the recipe-P models, but the most accurate configuration in the paper is
ConvNeXt-Base with recipe B. This notebook trains it again so its weights can be published (Notebook 06).

**Requires:** Notebooks 01 and 02 finished. **Runtime:** T4 GPU. **Expected time:** ≈ 45–55 min (5 folds × ≈ 9 min).
Safe to interrupt, exactly like Notebook 02: reconnect and **Runtime → Run all**; finished folds are skipped.

### What is identical to the reported run
* the training code (the same cells as Notebooks 02 and 03),
* the settings: read from the reported run's own `04_results/convnext_base__B/config.json`,
* the folds (`02_splits/cv_splits.csv`) and the random seed of every fold.

### What is different
* The weights of the selected epoch are saved: `06_models/convnext_base__B/r0_f0.pt … r0_f4.pt`.
* Results go to **separate** folders, `04_results_rerun/` and `05_checkpoints_rerun/`.
  **Nothing in `04_results/` is touched**, so every number in the paper stays as it is.

GPU training with mixed precision is not bit-for-bit repeatable, so the re-run will differ slightly from the reported
run. Step 6 measures how much (fold by fold, and with a McNemar test on the 1,720 paired predictions).

**When finished:** send me `NB05_results.zip` (downloaded by the last cell; a copy is in `04_results_rerun/`).

## 0. Settings

In [1]:
import os
NOTEBOOK, NB_TAG = "05_Retrain_ConvNeXtB_Weights", "NB05"
DRIVE_ROOT  = os.environ.get("PEANUT_DRIVE_ROOT", "/content/drive/MyDrive/Peanut_Revision")
LOCAL_ROOT  = os.environ.get("PEANUT_LOCAL_ROOT", "/content/peanut")

RERUN_CONFIGS = ["convnext_base__B"]
REPEATS      = [0]
FOLDS        = None     # None = all folds
CKPT_EVERY   = 5
NUM_WORKERS  = 2

TEST_MODE = os.environ.get("PEANUT_TEST") == "1"   # smoke test outside Colab only
if TEST_MODE:
    CKPT_EVERY, NUM_WORKERS, FOLDS = 2, 0, [0, 1]

## 1. Drive, GPU and environment

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys, json, time, shutil, zipfile, random, copy, platform, io, datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
import torchvision.transforms as T
import torchvision.models as M
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageEnhance, ImageFilter
import sklearn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

D = {k: Path(DRIVE_ROOT) / v for k, v in {"data": "01_data", "splits": "02_splits", "results": "04_results",
     "checkpoints": "05_checkpoints", "models": "06_models", "logs": "09_logs"}.items()}
for p in D.values(): p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
GPU = torch.cuda.get_device_name(0) if AMP else "CPU"
if not AMP and not TEST_MODE:
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then run all again.")
torch.backends.cudnn.benchmark = True

ENV = {"notebook": NOTEBOOK, "run_at": dt.datetime.now().isoformat(timespec="seconds"),
       "gpu": GPU, "python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "numpy": np.__version__, "scikit-learn": sklearn.__version__, "pillow": Image.__version__}
(D["logs"] / f"env_{NB_TAG}_{ENV['run_at'].replace(':', '-')}.json").write_text(json.dumps(ENV, indent=2))
print(json.dumps(ENV, indent=2))

Mounted at /content/drive
{
  "notebook": "05_Retrain_ConvNeXtB_Weights",
  "run_at": "2026-11-13T18:57:19",
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "torch": "2.14.1+cu130",
  "torchvision": "0.29.1+cu130",
  "cuda": "13.0",
  "cudnn": 92800,
  "numpy": "2.4.6",
  "scikit-learn": "1.6.1",
  "pillow": "11.3.0"
}


In [3]:
# Separate output folders: the reported results in 04_results/ are only read, never written.
ORIG_RESULTS = Path(DRIVE_ROOT) / "04_results"
D["results"] = Path(DRIVE_ROOT) / "04_results_rerun"
D["checkpoints"] = Path(DRIVE_ROOT) / "05_checkpoints_rerun"
for p in (D["results"], D["checkpoints"]): p.mkdir(parents=True, exist_ok=True)
for name in RERUN_CONFIGS:
    if not (ORIG_RESULTS / name / "config.json").exists():
        raise SystemExit(f"04_results/{name} not found: run Notebook 02 first.")
    old_w = sorted((D["models"] / name).glob("r*_f*.pt"))
    done = sorted((D["results"] / name).glob("r*_f*/metrics.json"))
    if old_w and not done:
        print(f"Note: 06_models/{name} already holds {len(old_w)} weight files without a finished re-run; they will be overwritten.")
print("reported results (read only):", ORIG_RESULTS)
print("re-run results:", D["results"])
print("weights:", D["models"])

reported results (read only): /content/drive/MyDrive/Peanut_Revision/04_results
re-run results: /content/drive/MyDrive/Peanut_Revision/04_results_rerun
weights: /content/drive/MyDrive/Peanut_Revision/06_models


## 2. Index, folds and images

In [4]:
CLASSES = ["Alternaria leaf spot", "Healthy", "Leaf spot (early and late)", "Rosette", "Rust"]
SHORT = ["Alternaria", "Healthy", "Leaf spot", "Rosette", "Rust"]

idx = pd.read_csv(D["splits"] / "index_original.csv")
cv = pd.read_csv(D["splits"] / "cv_splits.csv")
cv_meta = json.loads((D["splits"] / "cv_splits_meta.json").read_text())
print(f"{len(idx)} images | grouping: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, "
      f"or {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or combined (cos, pHash, dHash) {cv_meta.get('combo', '-')} "
      f"| {cv_meta['n_groups']} duplicate groups | {cv_meta['k']} folds")

# Guard: results in 04_results must come from the current splits.
sig_file = D["results"] / "split_signature.json"
if sig_file.exists() and json.loads(sig_file.read_text()) != cv_meta:
    raise SystemExit("04_results/ contains results trained on DIFFERENT splits.\n"
                     "Run Notebook 01 again (it archives old results automatically), then run this notebook again.")
sig_file.write_text(json.dumps(cv_meta, indent=2))

local_cache = Path(LOCAL_ROOT) / "cache448"
if not any(local_cache.rglob("*.jpg")):
    t0 = time.time()
    local_zip = Path(LOCAL_ROOT) / "groundnut_v2_cache448.zip"
    local_zip.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(D["data"] / "groundnut_v2_cache448.zip", local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(LOCAL_ROOT)
    local_zip.unlink()
    print(f"cache copied and unzipped in {time.time()-t0:.0f}s")

t0 = time.time()
IMAGES = {iid: np.asarray(Image.open(local_cache / rel).convert("RGB")) for iid, rel in zip(idx["image_id"], idx["cache_rel"])}
LABELS = dict(zip(idx["image_id"], idx["label"]))
print(f"{len(IMAGES)} images loaded into RAM in {time.time()-t0:.0f}s "
      f"({sum(a.nbytes for a in IMAGES.values())/1e9:.2f} GB)")

1720 images | grouping: pHash <= 12 and dHash <= 16, or dinov2_vits14 cosine >= 0.9694550037384033, or combined (cos, pHash, dHash) [0.95, 20, 20] | 1112 duplicate groups | 5 folds
cache copied and unzipped in 2s
1720 images loaded into RAM in 5s (1.04 GB)


## 3. Recipes and models (identical to Notebook 02)

In [5]:
@dataclass
class Config:
    name: str
    backbone: str
    recipe: str
    aug: str = "A"
    sampler: bool = True
    class_weighted_loss: bool = True
    label_smoothing: float = 0.0
    optimizer: str = "adam"
    lr: float = 1e-3
    weight_decay: float = 1e-5
    scheduler: str = "onecycle"
    epochs: int = 25
    finetune: str = "partial"
    head: str = "mlp"
    batch_size: int = 32
    img_size: int = 224
    select_metric: str = "val_macro_f1"
    pretrained: bool = True
    split: str = "cv_splits"          # which split file in 02_splits/ (cv_splits = group-aware)
    notes: str = ""

def recipe_P(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__P")
    return Config(backbone=backbone, recipe="P", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

def recipe_B(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__B")
    return Config(backbone=backbone, recipe="B", aug="B", sampler=False, class_weighted_loss=False,
                  label_smoothing=0.2, optimizer="adamw", lr=1e-5, weight_decay=1e-4, scheduler="cosine_wr",
                  finetune="full", head="linear", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

BACKBONES = ["resnet50", "efficientnet_b0", "efficientnet_b4", "convnext_base"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_transforms(aug, size):
    if aug == "A":    # Disease_CNN_Transfer_learning_RasNet_50.ipynb
        train = T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.ToTensor(), T.RandomErasing(p=0.2),
                           T.Normalize(MEAN, STD)])
    elif aug == "B":  # Disease-EfficientNet-B0/B4, Disease-ConvNeXt-Base
        train = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomRotation(15), T.RandomResizedCrop(size, scale=(0.7, 1.0)),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.RandomGrayscale(p=0.1), T.ToTensor(),
                           T.Normalize(MEAN, STD)])
    elif aug == "none":
        train = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    else:
        raise ValueError(aug)
    test = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    return train, test

def build_model(cfg, num_classes=5):
    def head(in_f):
        if cfg.head == "mlp":
            return nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))
        return nn.Linear(in_f, num_classes)
    b, pre = cfg.backbone, cfg.pretrained
    if b == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V1 if pre else None)
        m.fc = head(m.fc.in_features); trainable = ["layer4.", "fc."]
    elif b == "efficientnet_b0":
        m = M.efficientnet_b0(weights=M.EfficientNet_B0_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "efficientnet_b4":
        m = M.efficientnet_b4(weights=M.EfficientNet_B4_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "convnext_base":
        m = M.convnext_base(weights=M.ConvNeXt_Base_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[2] = head(m.classifier[2].in_features); trainable = ["features.6.", "features.7.", "classifier."]
    else:
        raise ValueError(b)
    if cfg.finetune == "partial":
        for n, p in m.named_parameters():
            p.requires_grad = any(n.startswith(t) for t in trainable)
    return m

def show_configs(configs):
    for cfg in configs:
        m = build_model(replace(cfg, pretrained=False))
        tr = sum(p.numel() for p in m.parameters() if p.requires_grad); tot = sum(p.numel() for p in m.parameters())
        print(f"{cfg.name:<36} trainable {tr/1e6:6.2f} M of {tot/1e6:6.2f} M | input {cfg.img_size} | batch {cfg.batch_size} | split {cfg.split}")
        del m

In [6]:
# ---- simulated field degradations (applied to the 448-px image before resizing) ----
def _noise(img, std, seed):
    a = np.asarray(img).astype(np.float32) / 255.0
    a = a + np.random.default_rng(seed).normal(0, std, a.shape)
    return Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))

def _occlude(img, frac, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).copy(); h, w = a.shape[:2]
    s = int(np.sqrt(frac) * min(h, w)); y, x = rng.integers(0, h - s), rng.integers(0, w - s)
    a[y:y + s, x:x + s] = 0
    return Image.fromarray(a)

def _shadow(img, factor, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).astype(np.float32); w = a.shape[1]
    x0 = rng.integers(0, w // 2); a[:, x0:x0 + w // 2] *= factor
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def _jpeg(img, q):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=q); buf.seek(0)
    return Image.open(buf).convert("RGB")

PERTURBATIONS = {
    "clean":            lambda im, s: im,
    "dark_0.6":         lambda im, s: ImageEnhance.Brightness(im).enhance(0.6),
    "overexposed_1.4":  lambda im, s: ImageEnhance.Brightness(im).enhance(1.4),
    "low_contrast_0.6": lambda im, s: ImageEnhance.Contrast(im).enhance(0.6),
    "hard_shadow_0.4":  lambda im, s: _shadow(im, 0.4, s),
    "blur_sigma2":      lambda im, s: im.filter(ImageFilter.GaussianBlur(2)),
    "noise_sd0.05":     lambda im, s: _noise(im, 0.05, s),
    "jpeg_q20":         lambda im, s: _jpeg(im, 20),
    "occlusion_25pct":  lambda im, s: _occlude(im, 0.25, s),
    "rotation_45":      lambda im, s: im.rotate(45),
}

## 4. Training code (identical to Notebook 02)

In [7]:
SPLITS = {"cv_splits": cv}            # notebooks can add more split tables (e.g. the naive split in Notebook 03)

class ImageSet(Dataset):
    def __init__(self, ids, tf, perturb=None):
        self.ids, self.tf, self.perturb = list(ids), tf, perturb
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.fromarray(IMAGES[self.ids[i]])
        if self.perturb is not None:
            img = self.perturb(img, i)
        return self.tf(img), LABELS[self.ids[i]]

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def metrics(y, p, prob=None):
    lab = list(range(len(CLASSES)))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=lab, zero_division=0)
    out = {"accuracy": accuracy_score(y, p), "n_correct": int((np.asarray(y) == np.asarray(p)).sum()), "n": int(len(y)),
           "balanced_accuracy": balanced_accuracy_score(y, p),
           "macro_precision": float(pr.mean()), "macro_recall": float(rc.mean()),
           "macro_f1": f1_score(y, p, average="macro", labels=lab, zero_division=0),
           "weighted_f1": f1_score(y, p, average="weighted", labels=lab, zero_division=0),
           "cohen_kappa": cohen_kappa_score(y, p),
           "per_class": {CLASSES[i]: {"precision": float(pr[i]), "recall": float(rc[i]), "f1": float(f1[i]),
                                      "support": int(sup[i])} for i in lab},
           "confusion_matrix": confusion_matrix(y, p, labels=lab).tolist()}
    if prob is not None and len(np.unique(y)) == len(lab):
        out["macro_auc_ovr"] = roc_auc_score(y, prob, multi_class="ovr", labels=lab)
    return {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in out.items()}

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, ps, probs, loss, n = [], [], [], 0.0, 0
    ce = nn.CrossEntropyLoss(reduction="sum")
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(x)
        out = out.float(); loss += ce(out, y).item(); n += len(y)
        probs.append(torch.softmax(out, 1).cpu().numpy()); ps.append(out.argmax(1).cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(probs), loss / max(n, 1)

def ckpt_path(cfg): return D["checkpoints"] / f"{cfg.name}.pt"

def save_checkpoint(cfg, r, f, epoch, model, opt, sched, scaler, sampler_gen, best, best_state, best_epoch, log):
    state = {"config": cfg.name, "repeat": r, "fold": f, "epoch": epoch, "model": model.state_dict(),
             "opt": opt.state_dict(), "sched": sched.state_dict() if sched else None, "scaler": scaler.state_dict(),
             "sampler_gen": sampler_gen.get_state(), "best": best, "best_epoch": best_epoch,
             "best_state": {k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()},
             "log": log, "rng": {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
                                 "cuda": torch.cuda.get_rng_state_all() if AMP else None}}
    torch.save(state, ckpt_path(cfg))      # same file every time: overwritten in place, never sent to the Drive bin
    if os.environ.get("PEANUT_CRASH_AFTER_EPOCH") == str(epoch):   # smoke-test hook only
        raise KeyboardInterrupt("simulated disconnect")

def load_checkpoint(cfg, r, f):
    p = ckpt_path(cfg)
    if not p.exists() or p.stat().st_size == 0: return None
    try:
        st = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print("  checkpoint unreadable, starting this fold from scratch:", e); return None
    if (st["config"], st["repeat"], st["fold"]) != (cfg.name, r, f): return None
    return st

def clear_checkpoint(cfg):
    p = ckpt_path(cfg)
    if p.exists() and p.stat().st_size > 0:
        open(p, "wb").close()             # empty it (keeps one 0-byte file instead of filling the bin)

def run_fold(cfg, r, f, robustness=False, save_weights=False):
    r, f = int(r), int(f)                 # plain Python ints (numpy ints break random.seed and JSON)
    run_dir = D["results"] / cfg.name / f"r{r}_f{f}"
    if (run_dir / "metrics.json").exists():
        print(f"[{cfg.name}] repeat {r} fold {f}: already done"); return json.loads((run_dir / "metrics.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    sp = SPLITS[cfg.split]
    s = sp[(sp["repeat"] == r) & (sp["fold"] == f)]
    ids = {role: s.loc[s["role"] == role, "image_id"].tolist() for role in ("train", "val", "test")}
    seed = 1000 * r + f                      # same seed for every configuration -> paired comparisons
    seed_everything(seed)
    tf_train, tf_test = get_transforms(cfg.aug, cfg.img_size)
    y_tr = np.array([LABELS[i] for i in ids["train"]])
    gen = torch.Generator().manual_seed(seed)
    if cfg.sampler:
        w = (1.0 / np.bincount(y_tr, minlength=len(CLASSES)))[y_tr]
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
                           sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), True, generator=gen),
                           pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    else:
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, shuffle=True, generator=gen,
                           num_workers=NUM_WORKERS, pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    dl_va = DataLoader(ImageSet(ids["val"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)
    dl_te = DataLoader(ImageSet(ids["test"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)

    model = build_model(cfg).to(DEVICE)
    cw = None
    if cfg.class_weighted_loss:
        cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_tr), dtype=torch.float, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=cfg.label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = (torch.optim.Adam if cfg.optimizer == "adam" else torch.optim.AdamW)(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    per_batch = cfg.scheduler == "onecycle"
    sched = {"onecycle": lambda: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, steps_per_epoch=len(dl_tr), epochs=cfg.epochs),
             "cosine_wr": lambda: torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2),
             "cosine": lambda: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs),
             "none": lambda: None}[cfg.scheduler]()
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    start, best, best_state, best_epoch, log = 0, (-1.0, -1.0), None, 0, []
    st = load_checkpoint(cfg, r, f)
    if st:
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        if sched and st["sched"]: sched.load_state_dict(st["sched"])
        gen.set_state(st["sampler_gen"]); best, best_epoch, log = tuple(st["best"]), st["best_epoch"], st["log"]
        best_state = {k: v.float() if v.is_floating_point() else v for k, v in st["best_state"].items()}
        random.setstate(st["rng"]["py"]); np.random.set_state(st["rng"]["np"]); torch.set_rng_state(st["rng"]["torch"])
        if AMP and st["rng"]["cuda"]: torch.cuda.set_rng_state_all(st["rng"]["cuda"])
        start = st["epoch"]
        print(f"  resumed from checkpoint after epoch {start}")
        del st

    print(f"[{cfg.name}] repeat {r} fold {f}: train {len(ids['train'])}, val {len(ids['val'])}, test {len(ids['test'])}")
    t_fold = time.time()
    for ep in range(start, cfg.epochs):
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if per_batch: sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        if sched is not None and not per_batch: sched.step()
        yv, pv, _, vloss = predict(model, dl_va); vm = metrics(yv, pv)
        row = {"epoch": ep + 1, "train_loss": tl / tn, "train_acc": tc / tn, "val_loss": vloss, "val_acc": vm["accuracy"],
               "val_macro_f1": vm["macro_f1"], "lr": opt.param_groups[0]["lr"], "time_s": time.time() - t0}
        log.append(row)
        score = (row[cfg.select_metric], -vloss)
        mark = ""
        if score > best:
            best, best_epoch, mark = score, ep + 1, " *"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep+1:02d}  loss {row['train_loss']:.4f}  train acc {row['train_acc']:.4f}  "
              f"val acc {row['val_acc']:.4f}  val F1 {row['val_macro_f1']:.4f}  ({row['time_s']:.0f}s){mark}")
        if CKPT_EVERY and (ep + 1) % CKPT_EVERY == 0 and ep + 1 < cfg.epochs:
            save_checkpoint(cfg, r, f, ep + 1, model, opt, sched, scaler, gen, best, best_state, best_epoch, log)

    model.load_state_dict(best_state)
    yt, pt, prob, _ = predict(model, dl_te)
    m = metrics(yt, pt, prob)
    m.update({"config": asdict(cfg), "repeat": r, "fold": f, "seed": seed, "best_epoch": best_epoch,
              "best_val_macro_f1": best[0], "best_val_loss": -best[1],
              "train_minutes": (time.time() - t_fold) / 60, "n_train": len(ids["train"]), "n_val": len(ids["val"]),
              "n_test": len(ids["test"]), "gpu": GPU, "split_meta": cv_meta if cfg.split == "cv_splits" else cfg.split,
              "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
              "n_total_params": sum(p.numel() for p in model.parameters())})
    pred = pd.DataFrame({"image_id": ids["test"], "y_true": yt, "y_pred": pt})
    for i in range(len(CLASSES)): pred[f"p_{i}"] = prob[:, i]

    if robustness:
        rob = {}
        for name, fn in PERTURBATIONS.items():
            dl = DataLoader(ImageSet(ids["test"], tf_test, perturb=fn), batch_size=64, num_workers=NUM_WORKERS)
            y_, p_, _, _ = predict(model, dl); mm = metrics(y_, p_)
            rob[name] = {"accuracy": mm["accuracy"], "n_correct": mm["n_correct"], "n": mm["n"], "macro_f1": mm["macro_f1"],
                         "per_class_recall": {c: mm["per_class"][c]["recall"] for c in CLASSES}}
        (run_dir / "robustness.json").write_text(json.dumps(rob, indent=2))
        m["robustness"] = {k: v["accuracy"] for k, v in rob.items()}
    if save_weights:
        wdir = D["models"] / cfg.name; wdir.mkdir(parents=True, exist_ok=True)
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()}, wdir / f"r{r}_f{f}.pt")

    pd.DataFrame(log).to_csv(run_dir / "log.csv", index=False)
    pred.to_csv(run_dir / "predictions.csv", index=False)
    (run_dir / "metrics.json").write_text(json.dumps(m, indent=2))   # written last = the fold counts as done
    clear_checkpoint(cfg)
    print(f"[{cfg.name}] repeat {r} fold {f}: TEST accuracy {m['accuracy']:.4f} ({m['n_correct']}/{m['n']}), "
          f"macro-F1 {m['macro_f1']:.4f} (best epoch {best_epoch}, {m['train_minutes']:.1f} min)")
    del model, opt; torch.cuda.empty_cache() if AMP else None
    return m

def run_configs(configs, robustness_for=lambda c: False, weights_for=lambda c: False):
    for cfg in configs:
        (D["results"] / cfg.name).mkdir(parents=True, exist_ok=True)
        (D["results"] / cfg.name / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
        sp = SPLITS[cfg.split]; folds_all = sorted(int(x) for x in sp["fold"].unique())
        t0 = time.time()
        for r in REPEATS:
            for f in (FOLDS if FOLDS is not None else folds_all):
                run_fold(cfg, r, f, robustness=robustness_for(cfg), save_weights=weights_for(cfg))
        done = all((D["results"] / cfg.name / f"r{r}_f{f}" / "metrics.json").exists() for r in REPEATS for f in folds_all)
        if done and ckpt_path(cfg).exists():
            ckpt_path(cfg).unlink()           # model finished: its (empty) checkpoint file is no longer needed
        print(f"==> {cfg.name} finished in {(time.time()-t0)/60:.1f} min\n")

## 5. Re-train with weight saving
The configuration is rebuilt from the reported run's own `config.json`, so every setting is the same.

In [8]:
RERUN = []
for name in RERUN_CONFIGS:
    orig_cfg = json.loads((ORIG_RESULTS / name / "config.json").read_text())
    cfg = Config(**orig_cfg)
    if not TEST_MODE:
        ref = {"convnext_base__B": recipe_B("convnext_base")}.get(name)
        if ref is not None and asdict(ref) != orig_cfg:
            raise SystemExit(f"The saved config of {name} differs from recipe B in this notebook. Stop and tell me.")
    RERUN.append(cfg)
    print(json.dumps(asdict(cfg), indent=1))
show_configs(RERUN)
run_configs(RERUN, robustness_for=lambda c: False, weights_for=lambda c: True)

{
 "name": "convnext_base__B",
 "backbone": "convnext_base",
 "recipe": "B",
 "aug": "B",
 "sampler": false,
 "class_weighted_loss": false,
 "label_smoothing": 0.2,
 "optimizer": "adamw",
 "lr": 1e-05,
 "weight_decay": 0.0001,
 "scheduler": "cosine_wr",
 "epochs": 25,
 "finetune": "full",
 "head": "linear",
 "batch_size": 32,
 "img_size": 224,
 "select_metric": "val_macro_f1",
 "pretrained": true,
 "split": "cv_splits",
 "notes": ""
}
convnext_base__B                     trainable  87.57 M of  87.57 M | input 224 | batch 32 | split cv_splits
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:02<00:00, 163MB/s]


[convnext_base__B] repeat 0 fold 0: train 1200, val 154, test 366
  ep 01  loss 1.4238  train acc 0.4817  val acc 0.6688  val F1 0.3943  (43s) *
  ep 02  loss 1.1209  train acc 0.7567  val acc 0.8896  val F1 0.7684  (14s) *
  ep 03  loss 0.9223  train acc 0.9058  val acc 0.9416  val F1 0.9103  (14s) *
  ep 04  loss 0.8413  train acc 0.9433  val acc 0.9610  val F1 0.9496  (14s) *
  ep 05  loss 0.8176  train acc 0.9525  val acc 0.9610  val F1 0.9496  (15s) *
  ep 06  loss 0.7821  train acc 0.9658  val acc 0.9740  val F1 0.9809  (16s) *
  ep 07  loss 0.7428  train acc 0.9783  val acc 0.9805  val F1 0.9857  (16s) *
  ep 08  loss 0.7290  train acc 0.9800  val acc 0.9935  val F1 0.9952  (15s) *
  ep 09  loss 0.7128  train acc 0.9875  val acc 0.9805  val F1 0.9857  (15s)
  ep 10  loss 0.7095  train acc 0.9883  val acc 0.9870  val F1 0.9905  (15s)
  ep 11  loss 0.7060  train acc 0.9858  val acc 0.9740  val F1 0.9815  (15s)
  ep 12  loss 0.7003  train acc 0.9883  val acc 0.9870  val F1 0.9905  

## 6. Re-run versus the reported run
Per fold: test accuracy, macro-F1, selected epoch and how many of the test predictions are the same.
Overall: McNemar exact test on the 1,720 paired predictions (reported run vs re-run). A small, non-significant
difference means the published weights behave like the model reported in the paper.

In [9]:
def load_runs(name):
    return [json.loads(p.read_text()) for p in sorted((D["results"] / name).glob("r*_f*/metrics.json"))]

def summary_table(names):
    rows = []
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        row = {"configuration": name, "folds": len(ms)}
        for k in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "balanced_accuracy", "cohen_kappa"]:
            v = 100 * np.array([m[k] for m in ms])
            row[k] = f"{v.mean():.2f} ± {v.std(ddof=1) if len(v) > 1 else 0:.2f}"
        for c, sname in zip(CLASSES, SHORT):
            row[f"recall {sname}"] = f"{100*np.mean([m['per_class'][c]['recall'] for m in ms]):.1f}"
        row["pooled correct"] = f"{sum(m['n_correct'] if 'n_correct' in m else int(np.trace(m['confusion_matrix'])) for m in ms)}/{sum(m['n_test'] for m in ms)}"
        row["min/fold"] = f"{np.mean([m['train_minutes'] for m in ms]):.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

def robustness_table(names):
    rows = []
    for name in names:
        ms = [m for m in load_runs(name) if "robustness" in m]
        if ms:
            rows.append({"configuration": name, **{k: f"{100*np.mean([m['robustness'][k] for m in ms]):.1f}" for k in ms[0]["robustness"]}})
    return pd.DataFrame(rows).set_index("configuration").T if rows else None

def write_research_log(names, notebook, extra_sections=()):
    # Every number is written with repr() = the exact double stored in metrics.json (no rounding).
    R = repr
    L = [f"# Research log — {notebook}", "", f"Run: {ENV['run_at']} | GPU: {ENV['gpu']} | torch {ENV['torch']} | torchvision {ENV['torchvision']}",
         f"Split rule: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, or same class and {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or same class and (cosine, pHash, dHash) within {cv_meta.get('combo', '-')}; {cv_meta['n_groups']} groups; {cv_meta['k']} folds; seed {cv_meta['seed']}",
         "", "All values below are written programmatically from metrics.json / robustness.json with full precision (Python repr).",
         "Mean and SD are computed over folds (SD with ddof = 1) from the per-fold values listed.", ""]
    keys = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "cohen_kappa", "macro_auc_ovr"]
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        L += [f"## {name}", "", "Config: `" + json.dumps(ms[0]["config"]) + "`", "",
              "| fold | n_train | n_val | n_test | correct | " + " | ".join(keys) + " | best_epoch | best_val_macro_f1 | train_minutes |",
              "|" + "---|" * (8 + len(keys)) ]
        for m in ms:
            corr = m.get("n_correct", int(np.trace(m["confusion_matrix"])))
            L.append(f"| {m['fold']} | {m['n_train']} | {m['n_val']} | {m['n_test']} | {corr} | " +
                     " | ".join(R(m[k]) if k in m else "-" for k in keys) +
                     f" | {m['best_epoch']} | {R(m.get('best_val_macro_f1', '-'))} | {R(m['train_minutes'])} |")
        L += ["", "**Across folds**", ""]
        for k in keys:
            v = [m[k] for m in ms if k in m]
            if v:
                L.append(f"- {k}: mean {R(float(np.mean(v)))}, SD {R(float(np.std(v, ddof=1))) if len(v) > 1 else '-'}, min {R(min(v))}, max {R(max(v))}")
        tot_c = sum(m.get("n_correct", int(np.trace(m["confusion_matrix"]))) for m in ms); tot_n = sum(m["n_test"] for m in ms)
        L.append(f"- pooled out-of-fold: {tot_c}/{tot_n} correct = {R(tot_c / tot_n)}")
        pooled_cm = np.sum([np.array(m["confusion_matrix"]) for m in ms], axis=0)
        L += ["", "Pooled confusion matrix (rows = true, columns = predicted; order " + ", ".join(SHORT) + "):", "```", str(pooled_cm.tolist()), "```", ""]
        L += ["Per-class (per fold: precision / recall / F1 / support):", ""]
        L.append("| fold | " + " | ".join(SHORT) + " |"); L.append("|" + "---|" * (1 + len(CLASSES)))
        for m in ms:
            L.append(f"| {m['fold']} | " + " | ".join(f"{R(m['per_class'][c]['precision'])} / {R(m['per_class'][c]['recall'])} / {R(m['per_class'][c]['f1'])} / {m['per_class'][c]['support']}" for c in CLASSES) + " |")
        L += ["", "Per-class recall across folds: " + "; ".join(f"{s}: mean {R(float(np.mean([m['per_class'][c]['recall'] for m in ms])))}" for c, s in zip(CLASSES, SHORT)), ""]
        if any("robustness" in m for m in ms):
            perts = list(ms[0]["robustness"])
            L += ["Robustness (test accuracy under each degradation, per fold):", "", "| degradation | " + " | ".join(f"fold {m['fold']}" for m in ms) + " | mean |",
                  "|" + "---|" * (2 + len(ms))]
            for pz in perts:
                v = [m["robustness"][pz] for m in ms]
                L.append(f"| {pz} | " + " | ".join(R(x) for x in v) + f" | {R(float(np.mean(v)))} |")
            L.append("")
    for title, body in extra_sections:
        L += [f"## {title}", "", body, ""]
    path = D["logs"] / f"research_log_{NB_TAG}.md"
    path.write_text("\n".join(L))
    print("research log written:", path)
    return path

def make_results_zip(names, notebook, summary_df):
    out = Path(LOCAL_ROOT) / f"{NB_TAG}_results.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in names:
            for p in (D["results"] / name).rglob("*"):
                if p.is_file() and p.suffix in {".json", ".csv"}:
                    z.write(p, f"04_results/{p.relative_to(D['results'])}")
        for p in D["logs"].glob(f"*{NB_TAG}*"):
            z.write(p, f"09_logs/{p.name}")
        for p in D["splits"].glob("*.json"):
            z.write(p, f"02_splits/{p.name}")
        if summary_df is not None:
            z.writestr("summary.csv", summary_df.to_csv(index=False))
    shutil.copyfile(out, D["results"] / out.name)
    print(f"Saved {D['results'] / out.name} ({out.stat().st_size/1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(out))

from scipy.stats import binomtest
R = repr
COMPARE, cmp_text = {}, []
for name in RERUN_CONFIGS:
    rows, P_o, P_n, Y = [], [], [], []
    folds_done = sorted(int(p.parent.name.split("_f")[1]) for p in (D["results"] / name).glob("r0_f*/metrics.json"))
    for f in folds_done:
        o = json.loads((ORIG_RESULTS / name / f"r0_f{f}" / "metrics.json").read_text())
        n = json.loads((D["results"] / name / f"r0_f{f}" / "metrics.json").read_text())
        po = pd.read_csv(ORIG_RESULTS / name / f"r0_f{f}" / "predictions.csv")
        pn = pd.read_csv(D["results"] / name / f"r0_f{f}" / "predictions.csv")
        if po["image_id"].tolist() != pn["image_id"].tolist():
            raise SystemExit(f"fold {f}: different test images in the two runs. Stop and tell me.")
        P_o.append(po["y_pred"].to_numpy()); P_n.append(pn["y_pred"].to_numpy()); Y.append(po["y_true"].to_numpy())
        rows.append({"fold": f, "acc. reported": o["accuracy"], "acc. re-run": n["accuracy"],
                     "correct reported": o["n_correct"], "correct re-run": n["n_correct"], "n": o["n"],
                     "macro-F1 reported": o["macro_f1"], "macro-F1 re-run": n["macro_f1"],
                     "best epoch reported": o["best_epoch"], "best epoch re-run": n["best_epoch"],
                     "same predictions": float((po["y_pred"] == pn["y_pred"]).mean())})
    T_ = pd.DataFrame(rows)
    P_o, P_n, Y = map(np.concatenate, (P_o, P_n, Y))
    b = int(((P_o == Y) & (P_n != Y)).sum()); c = int(((P_o != Y) & (P_n == Y)).sum())
    p_mc = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    acc_o, acc_n = T_["acc. reported"].to_numpy(), T_["acc. re-run"].to_numpy()
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0
    res = {"folds": folds_done,
           "reported_mean": float(acc_o.mean()), "reported_sd": sd(acc_o), "reported_pooled": int((P_o == Y).sum()),
           "rerun_mean": float(acc_n.mean()), "rerun_sd": sd(acc_n), "rerun_pooled": int((P_n == Y).sum()), "n": int(len(Y)),
           "rerun_macro_f1_mean": float(T_["macro-F1 re-run"].mean()), "rerun_macro_f1_sd": sd(T_["macro-F1 re-run"].to_numpy()),
           "same_predictions": float((P_o == P_n).mean()), "n_same": int((P_o == P_n).sum()),
           "reported_right_rerun_wrong": b, "reported_wrong_rerun_right": c, "mcnemar_p": float(p_mc),
           "per_fold": T_.to_dict("records")}
    COMPARE[name] = res
    show = T_.copy()
    for k in ["acc. reported", "acc. re-run", "macro-F1 reported", "macro-F1 re-run", "same predictions"]:
        show[k] = (100 * show[k]).map(lambda x: f"{x:.2f}")
    print(f"=== {name} ==="); display(show)
    print(f"Reported run : {100*res['reported_mean']:.2f} ± {100*res['reported_sd']:.2f} %  "
          f"(pooled {res['reported_pooled']}/{res['n']})")
    print(f"Re-run       : {100*res['rerun_mean']:.2f} ± {100*res['rerun_sd']:.2f} %  "
          f"(pooled {res['rerun_pooled']}/{res['n']}); macro-F1 {100*res['rerun_macro_f1_mean']:.2f} ± {100*res['rerun_macro_f1_sd']:.2f} %")
    print(f"Same prediction for {res['n_same']}/{res['n']} test images ({100*res['same_predictions']:.2f}%); "
          f"reported right & re-run wrong: {b}, reported wrong & re-run right: {c}; McNemar exact p = {p_mc:.4g}")
    cmp_text.append(f"### {name}\n\n```\n{json.dumps({k: (R(v) if isinstance(v, float) else v) for k, v in res.items() if k != 'per_fold'}, indent=1)}\n```\n\n"
                    + "| fold | acc reported | acc re-run | correct reported | correct re-run | n | macro-F1 reported | macro-F1 re-run | best epoch reported | best epoch re-run | same predictions |\n|" + "---|" * 11 + "\n"
                    + "\n".join(f"| {r['fold']} | {R(r['acc. reported'])} | {R(r['acc. re-run'])} | {r['correct reported']} | {r['correct re-run']} | {r['n']} | "
                                f"{R(r['macro-F1 reported'])} | {R(r['macro-F1 re-run'])} | {r['best epoch reported']} | {r['best epoch re-run']} | {R(r['same predictions'])} |"
                                for r in res["per_fold"]))

(D["results"] / "rerun_comparison.json").write_text(json.dumps(COMPARE, indent=2))
for name in RERUN_CONFIGS:
    n_w = len(list((D["models"] / name).glob("r0_f*.pt")))
    print(f"{name}: {n_w} weight files in 06_models/{name}")
SUMMARY = summary_table(RERUN_CONFIGS)
write_research_log(RERUN_CONFIGS, NOTEBOOK, extra_sections=[("Re-run versus reported run", "\n\n".join(cmp_text))])
make_results_zip(RERUN_CONFIGS, NOTEBOOK, SUMMARY)

=== convnext_base__B ===


,fold,acc. reported,acc. re-run,correct reported,correct re-run,n,macro-F1 reported,macro-F1 re-run,best epoch reported,best epoch re-run,same predictions
0,0,99.45,99.45,364,364,366,98.82,98.82,24,24,100.00
1,1,99.37,99.37,316,316,318,99.54,99.54,24,24,100.00
2,2,99.41,99.41,339,339,341,99.15,99.15,23,23,100.00
3,3,98.59,98.59,349,349,354,98.51,98.51,22,22,100.00
4,4,99.71,99.71,340,340,341,99.79,99.79,17,17,100.00


Reported run : 99.31 ± 0.42 %  (pooled 1708/1720)
Re-run       : 99.31 ± 0.42 %  (pooled 1708/1720); macro-F1 99.16 ± 0.52 %
Same prediction for 1720/1720 test images (100.00%); reported right & re-run wrong: 0, reported wrong & re-run right: 0; McNemar exact p = 1
convnext_base__B: 5 weight files in 06_models/convnext_base__B
research log written: /content/drive/MyDrive/Peanut_Revision/09_logs/research_log_NB05.md
Saved /content/drive/MyDrive/Peanut_Revision/04_results_rerun/NB05_results.zip (0.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>